# AI Detects Abnormal Events in Sequential Data

Chapter 4 replication using UNSW-NB15, DabLog, and DeepLog. Select **Runtime → Change runtime type → T4 GPU**, then run cells in order.

## 1. Clone, install, and verify

In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys

REPO_URL = "https://github.com/kobbysika/sequential-anomaly-detection.git"
BRANCH = "main"
REPO = Path("/content/sequential-anomaly-detection")
if not REPO.exists():
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO)], check=True)
os.chdir(REPO)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements-colab.txt"], check=True)

import tensorflow as tf
print("Python:", sys.version.split()[0])
print("TensorFlow:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))
subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-v"], check=True)

## 2. Mount Drive and validate the raw data

Place the four original headerless files UNSW-NB15_1.csv through UNSW-NB15_4.csv in MyDrive/UNSW-NB15/raw. Do not use the separate predefined training/testing CSV files.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

RAW_DIR = Path("/content/drive/MyDrive/UNSW-NB15/raw")
RAW_FILES = [RAW_DIR / f"UNSW-NB15_{i}.csv" for i in range(1, 5)]
missing = [str(path) for path in RAW_FILES if not path.is_file()]
if missing:
    raise FileNotFoundError("Missing raw files:\n" + "\n".join(missing))
print("All four raw files found.")

## 3. Reproduce the directional split

The supplied splitter creates flow_from_this.csv and flow_to_this.csv per IP. A marker prevents duplicate appends when rerunning this cell.

In [ ]:
DATASET = REPO / "datasets/processed/unsw-nb15"
MARKER = DATASET / ".split_complete"
if MARKER.exists():
    print("Reusing completed directional split.")
else:
    existing = list(DATASET.glob("**/*.csv")) if DATASET.exists() else []
    if existing:
        raise RuntimeError("Incomplete processed directory. Restart the runtime before retrying.")
    DATASET.mkdir(parents=True, exist_ok=True)
    subprocess.run([
        sys.executable, "legacy/data/unswnb15/split.py", *map(str, RAW_FILES),
        "--dir", str(DATASET), "--head", "-1"
    ], check=True)
    MARKER.write_text("complete\n")
print("Processed data:", DATASET)

## 4. Validate and preprocess

In [ ]:
PREPARED = REPO / "results/runs/shared/preprocessed"
subprocess.run([
    sys.executable, "-m", "sequential_anomaly.cli.prepare_data",
    "--config", "configs/pilot.yaml", "--output", str(PREPARED)
], check=True)
print((PREPARED / "summary.json").read_text())

## 5. One-epoch pilot

This verifies both complete pipelines. Pilot metrics are not final report results.

In [ ]:
for model_name in ("dablog", "deeplog"):
    checkpoint_dir = REPO / "artifacts/checkpoints/pilot" / model_name
    evaluation_dir = REPO / "results/runs/pilot" / model_name
    subprocess.run([
        sys.executable, "-m", "sequential_anomaly.cli.train",
        "--config", "configs/pilot.yaml", "--model", model_name,
        "--prepared", str(PREPARED), "--output", str(checkpoint_dir),
        "--epochs", "1", "--batch-size", "64", "--patience", "1"
    ], check=True)
    subprocess.run([
        sys.executable, "-m", "sequential_anomaly.cli.evaluate",
        "--config", "configs/pilot.yaml", "--model", model_name,
        "--prepared", str(PREPARED),
        "--checkpoint", str(checkpoint_dir / "best.keras"),
        "--output", str(evaluation_dir)
    ], check=True)

## 6. Full replication

Review the pilot first, then set RUN_FULL_EXPERIMENT to True. The full run preserves the supplied 32-epoch UNSW driver setting and model-specific batch sizes.

In [ ]:
RUN_FULL_EXPERIMENT = False
if RUN_FULL_EXPERIMENT:
    for model_name in ("dablog", "deeplog"):
        config = f"configs/{model_name}.yaml"
        checkpoint_dir = REPO / "artifacts/checkpoints" / model_name
        subprocess.run([
            sys.executable, "-m", "sequential_anomaly.cli.train",
            "--config", config, "--prepared", str(PREPARED),
            "--output", str(checkpoint_dir), "--epochs", "32"
        ], check=True)
        subprocess.run([
            sys.executable, "-m", "sequential_anomaly.cli.evaluate",
            "--config", config, "--prepared", str(PREPARED),
            "--checkpoint", str(checkpoint_dir / "best.keras")
        ], check=True)
else:
    print("Full experiment disabled until the pilot is reviewed.")

## 7. Compare metrics and charts

In [ ]:
from IPython.display import Image, display
for model_name in ("dablog", "deeplog"):
    evaluation = (REPO / f"results/runs/{model_name}/evaluation" if RUN_FULL_EXPERIMENT
                  else REPO / f"results/runs/pilot/{model_name}")
    metrics = json.loads((evaluation / "metrics.json").read_text())
    print(model_name.upper(), {k: metrics[k] for k in
          ("auroc", "auprc", "best_f1", "best_f1_threshold")})
    display(Image(filename=str(evaluation / "roc_curve.png")))
    display(Image(filename=str(evaluation / "precision_recall_curve.png")))

## 8. Download results

The archive contains metrics and plots, not the raw dataset.

In [ ]:
from google.colab import files
archive = shutil.make_archive("/content/sequential_anomaly_results", "zip",
                              root_dir=REPO, base_dir="results")
print("Created:", archive)
files.download(archive)